In [2]:
pip install requests


In [24]:
# prompt: colocar los resultados en una tabla
import requests
import pandas as pd
from datetime import datetime
from textblob import TextBlob

# Tu API key de NewsAPI
api_key = 'COLOCAR API KEY'

# Endpoint para buscar noticias
url = 'https://newsapi.org/v2/everything'

# Parámetros de búsqueda
params = {
    'q': 'inteligencia artificial',  # Tema de búsqueda
    'language': 'es',                # Idioma en español
    'sortBy': 'publishedAt',         # Orden por fecha
    'apiKey': api_key                # Tu clave
}

# Realizar la solicitud
response = requests.get(url, params=params)

# Procesar respuesta
if response.status_code == 200:
    data = response.json()
    articles = data['articles'][:10]  # Mostrar las primeras 20 noticias

    # Crear una lista de diccionarios para el DataFrame
    article_list = []
    for article in articles:
        article_data = {
            'Título': article['title'],
            'Fuente': article['source']['name'],
            'Publicado': article['publishedAt'],
            'Enlace': article['url']
        }
        article_list.append(article_data)

    # Crear el DataFrame
    df = pd.DataFrame(article_list)

    # Mostrar el DataFrame como una tabla
    print(df)
else:
    print(f"Error: {response.status_code}")


                                              Título             Fuente  \
0  Leganés 0 - 1 Barcelona | Resumen, goles y res...     Europapress.es   
1  Según tiktoker argentino el jugador más descon...         Nacion.com   
2                                        Autoedición         Www.abc.es   
3  Apple cumple 49 años: una mirada al presente d...      Isenacode.com   
4  Pepe Aguilar niega haber criticado a Claudia S...      El Financiero   
5  El Interferómetro de La Palma: la apuesta que ...        Eldiario.es   
6  realme 14 Pro 5G 256GB 8GB Suede Grey con 120W...  Blogdechollos.com   
7  Diana Morant: "Ahogan a la universidad pública...        Eldiario.es   
8  Microcentro cuenta cuentos de terror e histori...          La Nacion   
9  Perú capacita a profesional sanitario para pod...  Montevideo.com.uy   

              Publicado                                             Enlace  
0  2025-04-12T20:57:42Z  https://www.europapress.es/deportes/estadistic...  
1  2025-04-12T20:40:

In [36]:
#Metricas de Calidad
# API Key
api_key = 'COLOCAR API KEY'

# Endpoint y parámetros
url = 'https://newsapi.org/v2/everything'
params = {
    'q': 'inteligencia artificial',
    'language': 'es',
    'sortBy': 'publishedAt',
    'pageSize': 10,
    'apiKey': api_key
}

# Solicitud a la API
response = requests.get(url, params=params)

# Verificación de enlace funcional
def verificar_enlace(url):
    try:
        r = requests.get(url, timeout=5)
        return r.status_code == 200
    except:
        return False

# Análisis de sentimiento del título
def analizar_sentimiento(texto):
    if texto:
        blob = TextBlob(texto)
        return blob.sentiment.polarity  # entre -1 y 1
    return 0

# Procesar respuesta
if response.status_code == 200:
    data = response.json()
    articles = data['articles']

    # Lista para almacenar resultados
    resultados = []

    for article in articles:
        titulo = article['title']
        fuente = article['source']['name']
        publicado = article['publishedAt']
        enlace = article['url']

        # Métricas de calidad
        longitud_titulo = len(titulo) if titulo else 0
        dias_desde_publicacion = (datetime.now() - datetime.fromisoformat(publicado[:-1])).days
        sentimiento = analizar_sentimiento(titulo)
        enlace_ok = verificar_enlace(enlace)

        resultados.append({
            'Título': titulo,
            'Fuente': fuente,
            'Publicado': publicado,
            'Días desde publicación': dias_desde_publicacion,
            'Longitud del título': longitud_titulo,
            'Sentimiento del título': sentimiento,
            '¿Enlace funcional?': enlace_ok,
            'URL': enlace
        })

    # Convertir a DataFrame y mostrar
    df = pd.DataFrame(resultados)
    print("\n Noticias con métricas de calidad:\n")
    print(df[['Título', 'Fuente', 'Días desde publicación', 'Longitud del título',
              'Sentimiento del título', '¿Enlace funcional?']])

    # Diversidad de fuentes
    fuentes_unicas = df['Fuente'].nunique()
    print(f"\n Número de fuentes únicas: {fuentes_unicas}")
    print("\n Distribución de fuentes (%):")
    print(df['Fuente'].value_counts(normalize=True) * 100)

else:
    print(f"❌ Error: {response.status_code}")



 Noticias con métricas de calidad:

                                              Título             Fuente  \
0  Leganés 0 - 1 Barcelona | Resumen, goles y res...     Europapress.es   
1  Según tiktoker argentino el jugador más descon...         Nacion.com   
2                                        Autoedición         Www.abc.es   
3  Apple cumple 49 años: una mirada al presente d...      Isenacode.com   
4  Pepe Aguilar niega haber criticado a Claudia S...      El Financiero   
5  El Interferómetro de La Palma: la apuesta que ...        Eldiario.es   
6  realme 14 Pro 5G 256GB 8GB Suede Grey con 120W...  Blogdechollos.com   
7  Diana Morant: "Ahogan a la universidad pública...        Eldiario.es   
8  Microcentro cuenta cuentos de terror e histori...          La Nacion   
9  Perú capacita a profesional sanitario para pod...  Montevideo.com.uy   

   Días desde publicación  Longitud del título  Sentimiento del título  \
0                       1                   71                 

#Explicación de las Metricas

##Dias desde la Publicación
Mide si dicha información proporcionada es Actual

##Número de fuentes únicas
Mide si estas fuentes son únicas y confiables

##Sentimiento del titulo
Evalua si la noticia es positiva, negativa o neutral en base al contexto solicitado
